# Holdout externo LAB019–LAB021 con `theta_full`

Este notebook ejecuta un **holdout de réplicas sin ajuste** sobre LAB019–LAB021.

## Protocolo congelado

- `t0 = 21/09/2026 17:00`, aplicación de levadura.
- La muestra 1, tomada después de la primera nutrición y antes del inóculo, inicializa el modelo en `t=0`.
- La primera nutrición (0.4 g FDA + 1 g SFX) ya está contenida en ese estado inicial y no se aplica otra vez.
- La segunda nutrición se aplica como un salto de N de 0.14 kg m$^{-3}$ en su hora ejecutada.
- La cinética usa los 17 parámetros de `theta_natural_full.csv`; LAB019–LAB021 no participan en ningún ajuste.
- La capa de CO2 usa los parámetros naturales `C_full_theta_refit`, calibrados antes de esta campaña.
- El escenario principal obtiene la activación de la **química predicha**. La activación basada en química observada se reporta aparte como diagnóstico condicionado.
- El pico operacional alrededor de `t0` se marca inválido con una regla raw congelada y 90 min de recuperación. No se interpola.

El holdout químico puntúa G+F, G, F, YAN, glicerol y, como capa secundaria congelada, piruvato, acetaldehído y acetato. Oculyze queda fuera de las métricas cuantitativas porque no existe una conversión homologada a kg m$^{-3}$.


In [ ]:
from __future__ import annotations

import csv
import json
import math
import re
import sys
from dataclasses import replace
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
get_ipython().run_line_magic("matplotlib", "inline")
import numpy as np
import openpyxl
import pandas as pd
from IPython.display import Markdown, display

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "fermentation_model").exists())
FM = ROOT / "fermentation_model"
LAB2026 = FM / "laboratory_2026"
DATA_DIR = FM / "data" / "mem2026" / "LAB019-021"
RESULTS_DIR = LAB2026 / "results" / "lab019_021_theta_full_holdout"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

for path in (FM, LAB2026, FM / "pilot_2025"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from shared import run_new_must_glycerol_estimability_doe as base
from shared import run_secondary_joint_campaign_doe as joint
from shared import run_secondary_v2_model_evaluation as secondary_v2
from laboratory_2026 import run_estimability_historical_by_medium as natural_loader
from laboratory_2026 import run_co2_matrix_cross_validation_2026 as co2_cross

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 110, "font.size": 9.5})

LABS = ("LAB019", "LAB020", "LAB021")
FERMENTER = {"LAB019": "F1", "LAB020": "F2", "LAB021": "F3"}
MATCHED_ORIGINAL = {"LAB019": "LAB004", "LAB020": "LAB006", "LAB021": "LAB009"}
COLORS = {"LAB019": "#4c72b0", "LAB020": "#55a868", "LAB021": "#c44e52"}
T0 = pd.Timestamp("2026-09-21 17:00:00")
PULSE2_DT = {
    "LAB019": pd.Timestamp("2026-09-24 09:20:00"),
    "LAB020": pd.Timestamp("2026-09-24 16:40:00"),
    "LAB021": pd.Timestamp("2026-09-23 16:30:00"),
}
PULSE2_H = {lab: (stamp - T0).total_seconds() / 3600.0 for lab, stamp in PULSE2_DT.items()}
DELTA_N_KG_M3 = 0.14
GRID_H = 0.25
CO2_FACTOR_G_L_H_PER_SCCM = 44.0095 * 60.0 * 0.74 / (1000.0 * 24.16 * 2.0)
assert np.isclose(CO2_FACTOR_G_L_H_PER_SCCM, 0.0404391928807947)
print("ROOT:", ROOT)
print("t0:", T0, "| pulse 2 [h]:", PULSE2_H)

## Parámetros congelados y procedencia

In [ ]:
THETA_FULL_PATH = LAB2026 / "results" / "estimability_historical_natural" / "theta_natural_full.csv"
THETA_AUX_PATH = LAB2026 / "results" / "final_operational_doe_v2" / "theta_final_operational_v2.csv"
CO2_PARAM_PATH = LAB2026 / "results" / "co2_matrix_cross_validation_2026_full_theta_sccm_corrected_no_lab010" / "three_state_co2_parameters.csv"

theta_table = pd.read_csv(THETA_FULL_PATH)
theta_full = dict(zip(theta_table["parameter"], theta_table["theta"].astype(float)))
assert set(base.FULL17) == set(theta_full), (set(base.FULL17) - set(theta_full), set(theta_full) - set(base.FULL17))

theta_aux = pd.read_csv(THETA_AUX_PATH, index_col=0).iloc[:, 0].astype(float).to_dict()
theta_model = {**theta_aux, **theta_full}  # theta_full siempre prevalece en la capa cinética

co2_parameter_table = pd.read_csv(CO2_PARAM_PATH)
co2_parameter_table = co2_parameter_table[
    co2_parameter_table["state"].eq("C_full_theta_refit")
    & co2_parameter_table["calibration_matrix"].eq("natural")
].copy()
co2_params = dict(zip(co2_parameter_table["parameter"], co2_parameter_table["estimate"].astype(float)))
assert set(co2_cross.PARAMETER_BOUNDS).issubset(co2_params)

theta_table.to_csv(RESULTS_DIR / "theta_full_used.csv", index=False)
co2_parameter_table.to_csv(RESULTS_DIR / "co2_parameters_used.csv", index=False)

protocol_lock = {
    "holdout_labs": list(LABS),
    "fit_performed": False,
    "t0": str(T0),
    "initial_sample": "m1 post-nutrition/pre-inoculation, assigned to model t=0",
    "theta_full": str(THETA_FULL_PATH.relative_to(ROOT)),
    "secondary_parameters": str(THETA_AUX_PATH.relative_to(ROOT)),
    "co2_parameters": "C_full_theta_refit/natural",
    "first_nutrition": "contained in m1; not replayed",
    "second_nitrogen_increment_kg_m3": DELTA_N_KG_M3,
    "co2_conversion_g_l_h_per_sccm_at_2L": CO2_FACTOR_G_L_H_PER_SCCM,
    "co2_initial_artifact_rule": "raw Hampel max(3.5*MAD_sigma, 0.5 SCCM), search [-0.5, 0.75] h, plus 90 min recovery",
    "primary_activation": "predicted G+F drop >=5 g/L or predicted E rise >=2 g/L",
    "conditional_activation": "observed chemistry bracket; diagnostic only",
}
(RESULTS_DIR / "protocol_lock.json").write_text(json.dumps(protocol_lock, indent=2, ensure_ascii=False), encoding="utf-8")

display(theta_table)
display(co2_parameter_table[["parameter", "estimate", "active_bound"]])

## Carga química y control de calidad

In [ ]:
def numeric(value):
    try:
        out = float(value)
        return out if np.isfinite(out) else np.nan
    except (TypeError, ValueError):
        return np.nan


def load_offline_samples():
    path = DATA_DIR / "Mediciones-Offline-LAB019-021.xlsx"
    wb = openpyxl.load_workbook(path, read_only=True, data_only=True)
    rows, pending = [], []
    for row in wb["MedicionesOffline"].iter_rows(min_row=2, values_only=True):
        fecha, hora, lab, brix, density = row[0], row[1], row[2], row[5], row[6]
        if lab is None and brix is None and density is None:
            continue
        if lab is None:
            pending.append((brix, density))
            continue
        hh = hora.time() if isinstance(hora, datetime) else hora
        rows.append({
            "lab": str(lab), "sample": str(row[4]).strip(),
            "sample_timestamp": datetime.combine(fecha.date() if hasattr(fecha, "date") else fecha, hh),
            "Brix": numeric(brix), "density": numeric(density), "metadata_fallback": False,
        })
    wb.close()
    for lab, (brix, density) in zip(LABS, pending):
        rows.append({
            "lab": lab, "sample": "11", "sample_timestamp": datetime(2026, 9, 25, 8, 30),
            "Brix": numeric(brix), "density": numeric(density), "metadata_fallback": True,
        })
    return pd.DataFrame(rows)


def load_y15():
    rows = []
    paths = sorted(DATA_DIR.glob("EXPAuto*.txt")) + sorted(DATA_DIR.glob("ONLINE*.txt"))
    for path in paths:
        for line in open(path, encoding="utf-8-sig", errors="replace"):
            fields = line.rstrip("\n").split("\t")
            if len(fields) < 6:
                continue
            raw = fields[3].strip()
            value = float(raw.replace(",", ".")) if re.fullmatch(r"-?\d+(,\d+)?", raw) else np.nan
            rows.append({"sample_id": fields[0], "analyte": fields[1], "value": value, "raw_value": raw, "analysis_timestamp": fields[5], "source_file": path.name})
    long = pd.DataFrame(rows).drop_duplicates(["sample_id", "analyte"], keep="first")
    wide = long.pivot(index="sample_id", columns="analyte", values="value")
    raw_wide = long.pivot(index="sample_id", columns="analyte", values="raw_value")
    return long, wide, raw_wide


offline = load_offline_samples()
y15_long, y15, y15_raw = load_y15()

chem_rows = []
for lab in LABS:
    samples = offline[offline["lab"].eq(lab)].copy().set_index("sample")
    for sample in [str(i) for i in range(1, 12)]:
        sid = f"{lab}-{sample}"
        meta = samples.loc[sample]
        stamp = pd.Timestamp(meta["sample_timestamp"])
        sample_t_h = (stamp - T0).total_seconds() / 3600.0
        t_h = 0.0 if sample == "1" else sample_t_h
        get = lambda name: float(y15.loc[sid, name]) if sid in y15.index and name in y15.columns and pd.notna(y15.loc[sid, name]) else np.nan
        get_raw = lambda name: str(y15_raw.loc[sid, name]) if sid in y15_raw.index and name in y15_raw.columns and pd.notna(y15_raw.loc[sid, name]) else ""
        gf = get("GLU-FRU-320")
        glucose_raw = get("GLUCOSE-320")
        glucose = glucose_raw if np.isfinite(glucose_raw) and glucose_raw >= 0.0 else np.nan
        fructose = gf - glucose if np.isfinite(gf) and np.isfinite(glucose) and gf >= glucose else np.nan
        pan, ammonia_raw = get("PAN"), get("AMMONIA")
        ammonia = max(ammonia_raw, 0.0) if np.isfinite(ammonia_raw) else np.nan
        yan = pan + 0.823 * ammonia if np.isfinite(pan) and np.isfinite(ammonia) else np.nan
        chem_rows.append({
            "lab": lab, "sample": sample, "sample_timestamp": stamp, "sample_t_h": sample_t_h, "t_h": t_h,
            "metadata_fallback": bool(meta["metadata_fallback"]), "density": float(meta["density"]), "Brix": float(meta["Brix"]),
            "GF_g_l": gf, "G_g_l": glucose, "G_raw_g_l": glucose_raw, "F_g_l": fructose,
            "PAN_mg_l": pan, "NH3_raw_mg_l": ammonia_raw, "NH3_censored_mg_l": ammonia,
            "YAN_mg_l": yan, "N_kg_m3": yan / 1000.0 if np.isfinite(yan) else np.nan,
            "Gly_g_l": get("GLYCEROL"), "Pyr_mg_l": get("PYRUVIC ACID"),
            "AcAld_mg_l": get("ACETALDEHID-1000"), "Acetate_g_l": get("ACETIC ACID"),
            "G_raw_text": get_raw("GLUCOSE-320"), "GF_raw_text": get_raw("GLU-FRU-320"),
            "NH3_below_zero": bool(np.isfinite(ammonia_raw) and ammonia_raw < 0.0),
        })
chem = pd.DataFrame(chem_rows).sort_values(["lab", "t_h"]).reset_index(drop=True)
chem["is_initial"] = chem["sample"].eq("1")
chem["soft_flag_GF"] = chem["lab"].eq("LAB019") & chem["sample"].eq("4")
chem["soft_flag_metadata"] = chem["metadata_fallback"]

assert np.allclose(chem[chem["sample"].eq("1")]["t_h"], 0.0)
assert chem.query("lab == 'LAB020' and sample == '4'")["G_g_l"].isna().all()
assert chem.query("lab == 'LAB020' and sample == '4'")["G_raw_g_l"].iloc[0] == -7.28
chem.to_csv(RESULTS_DIR / "chemistry_observations_qc.csv", index=False, encoding="utf-8-sig")
display(chem[["lab", "sample", "sample_t_h", "t_h", "GF_g_l", "G_g_l", "F_g_l", "YAN_mg_l", "Gly_g_l", "Pyr_mg_l", "AcAld_mg_l", "NH3_below_zero", "metadata_fallback"]])

## Entradas operacionales y construcción de lotes

La biomasa inicial no puede obtenerse cuantitativamente de Oculyze sin una conversión homologada. Se heredan `X0` y `Xd0` del experimento original emparejado (LAB004/006/009), una decisión congelada e independiente del resultado de estas réplicas. La temperatura proviene del sensor de cada fermentador.


In [ ]:
def read_csv_window(path, usecols, start, end, chunksize=500_000):
    frames = []
    for chunk in pd.read_csv(path, usecols=usecols, chunksize=chunksize, low_memory=False):
        stamp = pd.to_datetime(chunk["timestamp"], errors="coerce")
        keep = stamp.between(start, end)
        if keep.any():
            part = chunk.loc[keep].copy()
            part["timestamp"] = stamp.loc[keep]
            frames.append(part)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=usecols)


window_start, window_end = T0 - pd.Timedelta(hours=6), T0 + pd.Timedelta(hours=170)
temperature_raw = read_csv_window(
    DATA_DIR / "backup_global_temperatura.csv", ["timestamp", "fermentador", "T", "SP"], window_start, window_end
)
for col in ("T", "SP"):
    temperature_raw[col] = pd.to_numeric(temperature_raw[col], errors="coerce")

co2_raw = read_csv_window(
    DATA_DIR / "backup_global_co2.csv", ["timestamp", "fermentador", "flow_sccm", "flow_filt_sccm", "status"], window_start, window_end
)
for col in ("flow_sccm", "flow_filt_sccm"):
    co2_raw[col] = pd.to_numeric(co2_raw[col], errors="coerce")

historical_data = natural_loader.load_historical_medium_data("natural")
historical_batches = {b.batch: b for b in joint.make_secondary_batches(historical_data)}
historical_secondary_initials = joint.secondary_initials_by_medium(historical_data)["natural"]

def temperature_profile(lab, grid):
    f = FERMENTER[lab]
    frame = temperature_raw[temperature_raw["fermentador"].eq(f)].dropna(subset=["T"]).sort_values("timestamp")
    t = (frame["timestamp"] - T0).dt.total_seconds().to_numpy() / 3600.0
    return np.interp(grid, t, frame["T"].to_numpy(dtype=float))


holdout_batches = {}
core_predictions = {}
secondary_predictions = {}
initial_rows = []
for lab in LABS:
    c = chem[chem["lab"].eq(lab)].sort_values("t_h")
    init = c[c["sample"].eq("1")].iloc[0]
    original = historical_batches[MATCHED_ORIGINAL[lab]]
    f = FERMENTER[lab]
    co2_lab = co2_raw[co2_raw["fermentador"].eq(f)]
    horizon = min(170.0, float((co2_lab["timestamp"].max() - T0).total_seconds() / 3600.0))
    grid = np.arange(0.0, horizon + GRID_H / 2.0, GRID_H)
    grid = np.unique(np.r_[grid, c["t_h"].to_numpy(dtype=float), PULSE2_H[lab]])
    grid = grid[(grid >= 0.0) & (grid <= horizon + 1e-9)]
    temp = temperature_profile(lab, grid)
    initials = {
        "X": float(original.initials["X"]), "Xd": float(original.initials["Xd"]),
        "N": float(init["N_kg_m3"]), "G": float(init["G_g_l"]), "F": float(init["F_g_l"]),
        "E": 0.0, "Gly": float(init["Gly_g_l"]),
        "Pyr": float(init["Pyr_mg_l"]),
        "AcAld": float(historical_secondary_initials["AcAld"]),
        "Acetate": float(init["Acetate_g_l"]),
        "O2": float(historical_secondary_initials["O2"]), "CO2": 0.0,
    }
    observations = {state: np.full(len(grid), np.nan) for state in base.STATE_NAMES}
    observations.update({state: np.full(len(grid), np.nan) for state in ("Pyr", "AcAld", "Acetate", "O2")})
    # Química observada sólo se almacena para el diagnóstico condicionado; no altera la simulación cinética.
    for row in c.itertuples():
        idx = int(np.argmin(np.abs(grid - float(row.t_h))))
        if abs(grid[idx] - float(row.t_h)) > 1e-7:
            continue
        for state, value in (("G", row.G_g_l), ("F", row.F_g_l), ("N", row.N_kg_m3), ("Gly", row.Gly_g_l), ("Pyr", row.Pyr_mg_l), ("AcAld", row.AcAld_mg_l), ("Acetate", row.Acetate_g_l)):
            observations[state][idx] = value
    batch = base.BatchData(
        medium="natural", batch=lab, time=grid, temperature_c=temp,
        pulses={"N": ((PULSE2_H[lab], DELTA_N_KG_M3),), "G": tuple(), "F": tuple(), "E": tuple(), "X": tuple()},
        observations=observations, initials=initials,
    )
    core = base.simulate(batch, theta_full, grid)
    assert core is not None and np.isfinite(core.to_numpy()).all()
    secondary = secondary_v2.integrate_secondary_v2(batch, theta_model, core)
    assert np.isfinite(secondary.to_numpy()).all()
    holdout_batches[lab], core_predictions[lab], secondary_predictions[lab] = batch, core, secondary
    initial_rows.append({"lab": lab, "matched_original_for_X0": MATCHED_ORIGINAL[lab], **initials, "horizon_h": horizon})

initial_table = pd.DataFrame(initial_rows)
initial_table.to_csv(RESULTS_DIR / "initial_conditions_used.csv", index=False)
display(initial_table)

## Holdout químico: predicciones y métricas

In [ ]:
CHEM_VARIABLES = {
    "GF": ("GF_g_l", "g/L"), "G": ("G_g_l", "g/L"), "F": ("F_g_l", "g/L"),
    "N": ("N_kg_m3", "kg/m3"), "Gly": ("Gly_g_l", "g/L"),
    "Pyr": ("Pyr_mg_l", "mg/L"), "AcAld": ("AcAld_mg_l", "mg/L"), "Acetate": ("Acetate_g_l", "g/L"),
}

chem_prediction_rows = []
for lab in LABS:
    core, secondary = core_predictions[lab], secondary_predictions[lab]
    pulse_h = PULSE2_H[lab]
    for row in chem[chem["lab"].eq(lab)].itertuples():
        if row.sample == "1":
            continue  # condición inicial, no observación de validación
        t = float(row.t_h)
        predictions = {
            "GF": float(np.interp(t, core.index, core["G"] + core["F"])),
            "G": float(np.interp(t, core.index, core["G"])),
            "F": float(np.interp(t, core.index, core["F"])),
            "N": float(np.interp(t, core.index, core["N"])),
            "Gly": float(np.interp(t, core.index, core["Gly"])),
            "Pyr": float(np.interp(t, secondary.index, secondary["Pyr"])),
            "AcAld": float(np.interp(t, secondary.index, secondary["AcAld"])),
            "Acetate": float(np.interp(t, secondary.index, secondary["Acetate"])),
        }
        for variable, (column, unit) in CHEM_VARIABLES.items():
            observed = getattr(row, column)
            if not np.isfinite(observed):
                continue
            soft_flag = bool(row.soft_flag_metadata or (variable == "GF" and row.soft_flag_GF))
            chem_prediction_rows.append({
                "lab": lab, "sample": row.sample, "t_h": t,
                "domain": "pre_pulse2" if t < pulse_h else "post_pulse2",
                "variable": variable, "unit": unit, "observed": float(observed),
                "predicted": predictions[variable], "residual": predictions[variable] - float(observed),
                "soft_flag": soft_flag, "used_primary": True,
            })

chem_predictions = pd.DataFrame(chem_prediction_rows)
chem_predictions.to_csv(RESULTS_DIR / "chemistry_prediction_rows.csv", index=False)

def metric_table(rows, exclude_soft=False):
    data = rows[~rows["soft_flag"]].copy() if exclude_soft else rows.copy()
    data = pd.concat([data, data.assign(domain="full")], ignore_index=True)
    output = []
    for keys, group in data.groupby(["lab", "variable", "unit", "domain"], sort=True):
        error = group["predicted"].to_numpy() - group["observed"].to_numpy()
        obs = group["observed"].to_numpy()
        denom = np.sum((obs - obs.mean()) ** 2)
        output.append({
            "lab": keys[0], "variable": keys[1], "unit": keys[2], "domain": keys[3], "n": len(group),
            "MAE": float(np.mean(np.abs(error))), "RMSE": float(np.sqrt(np.mean(error**2))),
            "bias_pred_minus_obs": float(np.mean(error)),
            "R2": float(1.0 - np.sum(error**2) / denom) if denom > 0 else np.nan,
        })
    return pd.DataFrame(output)

chem_metrics = metric_table(chem_predictions, exclude_soft=False)
chem_metrics_sensitivity = metric_table(chem_predictions, exclude_soft=True)
chem_metrics.to_csv(RESULTS_DIR / "chemistry_metrics.csv", index=False)
chem_metrics_sensitivity.to_csv(RESULTS_DIR / "chemistry_metrics_excluding_soft_flags.csv", index=False)
display(chem_metrics[chem_metrics["domain"].eq("full")].round(4))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8.2), sharex=True)
plot_variables = [("GF", "G+F [g/L]"), ("N", "N [kg/m3]"), ("Gly", "Glicerol [g/L]"), ("Pyr", "Piruvato [mg/L]"), ("AcAld", "Acetaldehído [mg/L]"), ("Acetate", "Acetato [g/L]")]
for ax, (variable, ylabel) in zip(axes.ravel(), plot_variables):
    for lab in LABS:
        rows = chem_predictions.query("lab == @lab and variable == @variable").sort_values("t_h")
        ax.scatter(rows["t_h"], rows["observed"], color=COLORS[lab], s=24, label=f"{lab} observado")
        if variable == "GF":
            pred = core_predictions[lab]["G"] + core_predictions[lab]["F"]
        elif variable in {"N", "Gly"}:
            pred = core_predictions[lab][variable]
        else:
            pred = secondary_predictions[lab][variable]
        ax.plot(pred.index, pred.values, color=COLORS[lab], lw=1.3, alpha=0.85)
        ax.axvline(PULSE2_H[lab], color=COLORS[lab], ls="--", lw=0.7, alpha=0.35)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("t desde inoculación [h]")
axes[0, 0].legend(ncol=3, fontsize=7)
fig.suptitle("Holdout químico LAB019–LAB021 — theta_full congelado")
fig.tight_layout(rect=(0, 0, 1, 0.96))
display(fig)
plt.close(fig)

## CO2: máscara operacional, observaciones horarias y offset de cero

In [ ]:
co2_hourly_parts, mask_rows = [], []
for lab in LABS:
    f = FERMENTER[lab]
    frame = co2_raw[co2_raw["fermentador"].eq(f)].sort_values("timestamp").copy()
    frame["t_h"] = (frame["timestamp"] - T0).dt.total_seconds() / 3600.0
    baseline = frame[frame["t_h"].between(-2.0, -0.5)]["flow_sccm"].dropna().to_numpy(dtype=float)
    baseline_center = float(np.median(baseline))
    sigma = float(1.4826 * np.median(np.abs(baseline - baseline_center)))
    threshold = max(3.5 * sigma, 0.5)
    search = frame[frame["t_h"].between(-0.5, 0.75)].copy()
    crossing = search[np.abs(search["flow_sccm"] - baseline_center) > threshold]
    if crossing.empty:
        event_start_h, event_raw_end_h = 0.0, 1.0
    else:
        event_start_h, event_raw_end_h = float(crossing["t_h"].min()), float(crossing["t_h"].max())
    mask_end_h = max(0.0, event_raw_end_h) + 1.5

    zero_support = frame[frame["t_h"].between(mask_end_h, 12.0)]["flow_filt_sccm"].dropna()
    zero_sccm = max(0.0, float(zero_support.quantile(0.10))) if len(zero_support) else 0.0
    frame["flow_corrected_sccm"] = np.maximum(frame["flow_filt_sccm"] - zero_sccm, 0.0)
    hourly = frame.set_index("timestamp")[["flow_corrected_sccm", "flow_filt_sccm", "flow_sccm"]].resample("1h").median().dropna(subset=["flow_corrected_sccm"]).reset_index()
    hourly["t_h"] = (hourly["timestamp"] - T0).dt.total_seconds() / 3600.0
    hourly = hourly[hourly["t_h"].between(0.0, 170.0)].copy()
    hourly["lab"] = lab
    hourly["co2_observed_g_l_h"] = hourly["flow_corrected_sccm"] * CO2_FACTOR_G_L_H_PER_SCCM
    hourly["co2_valid"] = hourly["t_h"] > mask_end_h
    hourly["left_censored"] = hourly["co2_observed_g_l_h"] < co2_cross.CO2_DETECTION_LIMIT_G_L_H
    co2_hourly_parts.append(hourly)
    mask_rows.append({
        "lab": lab, "fermenter": f, "baseline_raw_sccm": baseline_center, "baseline_sigma_sccm": sigma,
        "artifact_threshold_sccm": threshold, "event_start_h": event_start_h,
        "event_raw_end_h": event_raw_end_h, "recovery_tail_min": 90, "mask_end_h": mask_end_h,
        "sensor_zero_sccm": zero_sccm,
    })

co2_hourly = pd.concat(co2_hourly_parts, ignore_index=True)
mask_audit = pd.DataFrame(mask_rows)
co2_hourly.to_csv(RESULTS_DIR / "co2_hourly_observations.csv", index=False)
mask_audit.to_csv(RESULTS_DIR / "co2_mask_audit.csv", index=False)
display(mask_audit.round(4))

## Capa de CO2 congelada

Se generan dos escenarios sin optimización:

1. **Primary end-to-end:** el bracket de activación se obtiene de G+F/E predichos con `theta_full`.
2. **Observed-chemistry conditioned:** usa el bracket observado y sólo diagnostica la capa de CO2.

El onset observado de CO2 nunca mueve ni alinea una predicción.


In [ ]:
def activity_bracket(time_h, sugar, ethanol):
    time_h = np.asarray(time_h, dtype=float)
    sugar = np.asarray(sugar, dtype=float)
    ethanol = np.asarray(ethanol, dtype=float)
    valid = np.isfinite(sugar) | np.isfinite(ethanol)
    indices = np.flatnonzero(valid)
    if not len(indices):
        return {"lower_h": 0.0, "upper_h": 0.0, "center_h": 0.0, "scale_h": 1.0}
    s0 = sugar[np.isfinite(sugar)][0] if np.isfinite(sugar).any() else np.nan
    e0 = ethanol[np.isfinite(ethanol)][0] if np.isfinite(ethanol).any() else np.nan
    active = (np.isfinite(sugar) & (sugar <= s0 - 5.0)) | (np.isfinite(ethanol) & (ethanol >= e0 + 2.0))
    active_indices = np.flatnonzero(active)
    if len(active_indices):
        upper_i = int(active_indices[0])
        earlier = indices[indices < upper_i]
        lower_i = int(earlier[-1]) if len(earlier) else upper_i
        lower, upper = float(time_h[lower_i]), float(time_h[upper_i])
    else:
        lower = upper = float(time_h[indices[0]])
    return {"lower_h": lower, "upper_h": upper, "center_h": 0.5 * (lower + upper), "scale_h": max(1.0, (upper - lower) / (2.0 * math.log(9.0))) if upper > lower else 1.0}


co2_predictions = {}
activation_rows = []
for lab in LABS:
    batch = holdout_batches[lab]
    observed = co2_hourly[co2_hourly["lab"].eq(lab)].copy()
    horizon = float(observed["t_h"].max())
    stub = observed[["t_h"]].copy()
    stub["batch"] = lab
    stub["matrix"] = "natural"
    stub["calibratable"] = True
    stub["chemistry_last_h"] = horizon
    caches, _diag = co2_cross.build_driver_cache({lab: batch}, {"natural": theta_full}, stub)
    conditional_cache = caches[lab]

    core = core_predictions[lab]
    predicted_bracket = activity_bracket(core.index, core["G"] + core["F"], core["E"])
    c = chem[chem["lab"].eq(lab)].sort_values("t_h")
    observed_bracket = activity_bracket(c["t_h"], c["GF_g_l"], np.full(len(c), np.nan))
    primary_cache = replace(
        conditional_cache,
        chemical_activity_lower_h=predicted_bracket["lower_h"],
        chemical_activity_upper_h=predicted_bracket["upper_h"],
        chemical_activity_center_h=predicted_bracket["center_h"],
        chemical_activity_scale_h=predicted_bracket["scale_h"],
    )
    conditional_cache = replace(
        conditional_cache,
        chemical_activity_lower_h=observed_bracket["lower_h"],
        chemical_activity_upper_h=observed_bracket["upper_h"],
        chemical_activity_center_h=observed_bracket["center_h"],
        chemical_activity_scale_h=observed_bracket["scale_h"],
    )

    for scenario, cache, bracket in (
        ("primary_end_to_end", primary_cache, predicted_bracket),
        ("observed_chemistry_conditioned", conditional_cache, observed_bracket),
    ):
        qgas, qprod, o2, phi = co2_cross.raw_qgas_grid_prediction(
            cache,
            k_release_h=co2_params["kCO2_release_h"], sat_scale=co2_params["CO2sat_scale"],
            o2_qmax_mg_gdw_h=co2_params["O2_qmax_mg_gdw_h"], o2_initial_scale=co2_params["O2_initial_scale"],
            chemistry_aligned=True, nitrogen_boost_transition=True,
            pulse_t_rise_h=co2_params["pulse_t_rise_h"], pulse_activity_gain=co2_params["pulse_activity_gain"],
            continuous_release=True, bounded_chemical_activation=True,
            chem_activation_start_fraction=co2_params["chem_activation_start_fraction"],
            chem_activation_duration_fraction=co2_params["chem_activation_duration_fraction"],
        )
        co2_predictions[(scenario, lab)] = pd.DataFrame({
            "t_h": cache.time_h, "co2_predicted_g_l_h": co2_params["matrix_gain"] * qgas,
            "co2_produced_g_l_h": qprod, "O2_mg_l": o2, "anaerobic_fraction": phi,
        })
        activation_rows.append({"scenario": scenario, "lab": lab, **bracket})

activation_table = pd.DataFrame(activation_rows)
activation_table.to_csv(RESULTS_DIR / "activation_brackets.csv", index=False)
display(activation_table.round(3))

In [ ]:
def sustained_onset(time_h, values, threshold, consecutive=3):
    t, y = np.asarray(time_h, float), np.asarray(values, float)
    above = np.isfinite(y) & (y >= threshold)
    for i in range(0, len(y) - consecutive + 1):
        if above[i:i + consecutive].all():
            return float(t[i])
    return np.nan


prediction_rows, metric_rows = [], []
for lab in LABS:
    obs_all = co2_hourly[(co2_hourly["lab"].eq(lab)) & co2_hourly["co2_valid"]].sort_values("t_h").copy()
    early = obs_all[obs_all["t_h"].le(12.0)]
    baseline = float(early["co2_observed_g_l_h"].median()) if len(early) else 0.0
    peak = float(obs_all["co2_observed_g_l_h"].max())
    onset_threshold = max(co2_cross.CO2_DETECTION_LIMIT_G_L_H, baseline + 0.10 * max(peak - baseline, 0.0))
    for scenario in ("primary_end_to_end", "observed_chemistry_conditioned"):
        pred_curve = co2_predictions[(scenario, lab)]
        obs = obs_all.copy()
        obs["predicted"] = np.interp(obs["t_h"], pred_curve["t_h"], pred_curve["co2_predicted_g_l_h"])
        obs["scenario"] = scenario
        prediction_rows.append(obs[["scenario", "lab", "timestamp", "t_h", "co2_observed_g_l_h", "predicted", "left_censored"]])
        for domain, candidate in (
            ("pre_pulse2", obs["t_h"] < PULSE2_H[lab]),
            ("post_pulse2", obs["t_h"] >= PULSE2_H[lab]),
            ("full", np.ones(len(obs), dtype=bool)),
        ):
            part = obs[candidate].copy()
            if part.empty:
                continue
            error = part["predicted"].to_numpy() - part["co2_observed_g_l_h"].to_numpy()
            observed_values = part["co2_observed_g_l_h"].to_numpy()
            predicted_values = part["predicted"].to_numpy()
            times = part["t_h"].to_numpy()
            denom = np.sum((observed_values - observed_values.mean()) ** 2)
            dt = np.diff(times)
            pair_ok = (dt > 0) & (dt <= 1.5)
            auc_obs = float(np.sum(0.5 * (observed_values[:-1] + observed_values[1:]) * dt * pair_ok))
            auc_pred = float(np.sum(0.5 * (predicted_values[:-1] + predicted_values[1:]) * dt * pair_ok))
            onset_obs = sustained_onset(times, observed_values, onset_threshold)
            onset_pred = sustained_onset(times, predicted_values, onset_threshold)
            metric_rows.append({
                "scenario": scenario, "lab": lab, "domain": domain, "n": len(part),
                "MAE_g_l_h": float(np.mean(np.abs(error))), "RMSE_g_l_h": float(np.sqrt(np.mean(error**2))),
                "bias_pred_minus_obs_g_l_h": float(np.mean(error)),
                "R2": float(1.0 - np.sum(error**2) / denom) if denom > 0 else np.nan,
                "onset_threshold_g_l_h": onset_threshold, "onset_observed_h": onset_obs, "onset_predicted_h": onset_pred,
                "onset_error_h": onset_pred - onset_obs if np.isfinite(onset_obs) and np.isfinite(onset_pred) else np.nan,
                "peak_observed_g_l_h": float(observed_values.max()), "peak_predicted_g_l_h": float(predicted_values.max()),
                "peak_time_observed_h": float(times[np.argmax(observed_values)]), "peak_time_predicted_h": float(times[np.argmax(predicted_values)]),
                "AUC_observed_g_l": auc_obs, "AUC_predicted_g_l": auc_pred, "AUC_error_g_l": auc_pred - auc_obs,
            })

co2_prediction_rows = pd.concat(prediction_rows, ignore_index=True)
co2_metrics = pd.DataFrame(metric_rows)
co2_prediction_rows.to_csv(RESULTS_DIR / "co2_prediction_rows.csv", index=False)
co2_metrics.to_csv(RESULTS_DIR / "co2_metrics.csv", index=False)
display(co2_metrics[co2_metrics["domain"].eq("full")].round(4))

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13.5, 9.5), sharex=True)
for ax, lab in zip(axes, LABS):
    all_obs = co2_hourly[co2_hourly["lab"].eq(lab)].sort_values("t_h")
    valid = all_obs["co2_valid"]
    ax.plot(all_obs["t_h"], all_obs["co2_observed_g_l_h"], color="0.75", lw=1.0, label="observado (incluye zona enmascarada)")
    ax.scatter(all_obs.loc[valid, "t_h"], all_obs.loc[valid, "co2_observed_g_l_h"], color="black", s=10, label="observado válido")
    for scenario, color, label in (
        ("primary_end_to_end", COLORS[lab], "predicción end-to-end"),
        ("observed_chemistry_conditioned", "#dd8452", "diagnóstico condicionado por química"),
    ):
        pred = co2_predictions[(scenario, lab)]
        ax.plot(pred["t_h"], pred["co2_predicted_g_l_h"], color=color, lw=1.5, label=label)
    mask_end = float(mask_audit.loc[mask_audit["lab"].eq(lab), "mask_end_h"].iloc[0])
    ax.axvspan(0.0, mask_end, color="#d62728", alpha=0.12, label="agitación + recuperación")
    ax.axvline(PULSE2_H[lab], color="purple", ls="--", lw=1.0, label="nutrición 2")
    ax.set_ylabel(f"{lab}\nCO2 [g L$^{{-1}}$ h$^{{-1}}$]")
axes[0].legend(ncol=3, fontsize=8)
axes[-1].set_xlabel("t desde inoculación [h]")
fig.suptitle("Holdout de la capa de CO2 — theta_full y parámetros CO2 congelados")
fig.tight_layout(rect=(0, 0, 1, 0.97))
display(fig)
plt.close(fig)

## Verificaciones finales y lectura del resultado

In [ ]:
assert protocol_lock["fit_performed"] is False
assert set(activation_table.query("scenario == 'observed_chemistry_conditioned'")["upper_h"].round(6)) == {16.0}
assert len(chem_metrics) > 0 and len(co2_metrics) > 0
assert not chem_predictions["predicted"].isna().any()
assert not co2_prediction_rows["predicted"].isna().any()

primary_co2 = co2_metrics.query("scenario == 'primary_end_to_end' and domain == 'full'")
conditioned_co2 = co2_metrics.query("scenario == 'observed_chemistry_conditioned' and domain == 'full'")
chem_full = chem_metrics[chem_metrics["domain"].eq("full")]

summary_text = f'''
### Ejecución terminada

- No se ejecutó ningún ajuste ni optimizador.
- Se evaluaron **{len(chem_predictions)}** observaciones químicas válidas posteriores a la condición inicial.
- Se evaluaron **{int(primary_co2['n'].sum())}** observaciones horarias válidas de CO2 en el escenario principal.
- El bracket observado fue `[0, 16 h]` en los tres laboratorios; sólo se usa en el diagnóstico condicionado.
- La tabla principal de química es `chemistry_metrics.csv` y la de CO2 es `co2_metrics.csv`.

El escenario `primary_end_to_end` es el resultado de holdout. El escenario `observed_chemistry_conditioned` sirve para separar error de activación/dinámica upstream de error en la capa gaseosa.
'''
display(Markdown(summary_text))
display(chem_full.pivot_table(index=["lab", "variable"], values=["MAE", "RMSE", "bias_pred_minus_obs"]).round(4))
display(primary_co2[["lab", "RMSE_g_l_h", "onset_observed_h", "onset_predicted_h", "peak_observed_g_l_h", "peak_predicted_g_l_h", "AUC_observed_g_l", "AUC_predicted_g_l"]].round(4))
print("Notebook completed; optimization performed:", False)